# Citation network

Default paths are relative: inputs under `data/`, results under `outputs/05_citation_network/`. Preserve the private source input directory layout. Generated edge lists, community files, numeric tables and figures go to the configured results directory and are not included with the code.

Default execution reuses the saved Louvain communities; it never changes the original inputs. Set `REGENERATE_COMMUNITIES` explicitly to regenerate the source resolution grid with seed 42 (requires python-louvain). Louvain uses the undirected projection; downstream citation analysis uses directed, deduplicated edges. Main resolutions remain Normal 1.0 and Predatory 1.8; the source common-resolution sensitivity is retained. Target-sharing groups are connected components, not all-pairs similarity cliques. AI coloring excludes missing/unresolved saved decisions and shows unavailable values in gray.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '05_citation_network'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
import shutil
import pandas as pd
REGENERATE_COMMUNITIES = False
DRAW_SELECTED_COMMUNITY = False
for group in ['pred','norm']:
    (OUTPUT_DIR/group/'metadata/louvain').mkdir(parents=True,exist_ok=True)
if not REGENERATE_COMMUNITIES:
    for group,filename in [('norm','best_louvain_reso_1.csv'),('pred','best_louvain_reso_1.8.csv'),('pred','best_louvain_reso_1.csv')]:
        shutil.copyfile(DATA_DIR/group/'metadata/louvain'/filename,OUTPUT_DIR/group/'metadata/louvain'/filename)


<h1>Step 0: Convert citation table to citation edgelist</h1>

In [ ]:
import csv
def convert_csv_to_edgelist(input_csv,output_txt):
    Path(output_txt).parent.mkdir(parents=True,exist_ok=True)
    edges = pd.read_csv(input_csv,usecols=['citing_paper_id','cited_paper_id'])
    if edges.isna().any().any():
        raise ValueError('Citation endpoints must be nonmissing.')
    if not (edges.to_numpy()==edges.to_numpy().astype('int64')).all():
        raise ValueError('Citation endpoints must be integer identifiers.')
    edges = edges.drop_duplicates()
    edges.to_csv(output_txt,sep=' ',header=False,index=False)


In [ ]:
convert_csv_to_edgelist(str(DATA_DIR / 'norm/dataset/csv/citations.csv'), str(OUTPUT_DIR / 'norm/metadata/edge_list.txt'))
convert_csv_to_edgelist(str(DATA_DIR / 'pred/dataset/csv/citations.csv'), str(OUTPUT_DIR / 'pred/metadata/edge_list.txt'))

<h1>Step 1: Louvain Clustering</h1>

Note: To reproduce the analysis from the saved assignments, this step can be skipped and the Louvain results in `data/[norm/pred]/metadata/louvain` can be used directly. When communities are regenerated, Louvain uses the prespecified random seed 42 for both datasets. The seed is fixed for reproducibility and was not selected after inspecting downstream group differences.

In [ ]:
if REGENERATE_COMMUNITIES:
    from community import community_louvain
import networkx as nx
from collections import defaultdict
import numpy as np
import random
import csv
import os

LOUVAIN_RANDOM_SEED = 42


# -------------------------------
# Logger for console + file output
# -------------------------------
class Logger:
    def __init__(self, log_file):
        self.log_file = log_file
        with open(self.log_file, 'w') as f:
            f.write("")

    def log(self, message):
        print(message)
        with open(self.log_file, 'a') as f:
            f.write(message + "\n")


# -------------------------------
# Load edges from file
# -------------------------------
def load_citation_edges(file_path):
    edges = []
    with open(file_path, 'r') as f:
        for line in f:
            src, dst = map(int, line.strip().split())
            edges.append((src, dst))
    return edges


# -------------------------------
# Save clustering results to CSV
# -------------------------------
def save_results_to_csv(resolution, community_dict, modularity_value, output_file):
    with open(output_file, 'a', newline='') as csvfile:
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow([f"Resolution: {resolution}", f"Modularity: {modularity_value}"])
        csvwriter.writerow(["Community ID", "Number of Nodes", "Nodes"])
        for community_id, nodes in community_dict.items():
            csvwriter.writerow([community_id, len(nodes), ', '.join(map(str, nodes))])
        csvwriter.writerow([])  # blank line
    logger.log(f"Results for resolution {resolution} saved to {output_file}")


# -------------------------------
# Build NetworkX graph
# -------------------------------
def build_graph(edges):
    G = nx.Graph()
    G.add_edges_from(edges)
    return G


# -------------------------------
# Louvain clustering (reproducible)
# -------------------------------
def perform_louvain_clustering(G, resolution=1.0, seed=LOUVAIN_RANDOM_SEED):
    # random_state ensures reproducibility
    partition = community_louvain.best_partition(G, resolution=resolution, random_state=seed)
    return partition


# -------------------------------
# Analyze and log community info
# -------------------------------
def analyze_communities(partition):
    community_dict = defaultdict(list)
    for node, community_id in partition.items():
        community_dict[community_id].append(node)

    for community_id, nodes in community_dict.items():
        logger.log(f"Community {community_id}: {len(nodes)} nodes")

        logger.log("-" * 40)

    return community_dict


# -------------------------------
# Compute modularity score
# -------------------------------
def calculate_modularity_score(G, partition):
    # Modularity is the objective Louvain optimizes and the standard quality
    # measure for community detection; it is O(E) and needs no distance matrix.
    try:
        score = community_louvain.modularity(partition, G)
    except (ValueError, ZeroDivisionError) as e:
        score = None
        logger.log(f"Modularity calculation failed: {e}")

    return score


# -------------------------------
# Main pipeline
# -------------------------------
def main():
    global logger

    # fix all random sources (same seed for both datasets, for reproducibility)
    SEED = LOUVAIN_RANDOM_SEED

    # Run Louvain for BOTH datasets in one pass so a fresh Run-All regenerates
    # every clustering_results.csv the downstream steps need. Previously only
    # "norm" was hard-coded here, so the "pred" file had to be produced by
    # manually editing these paths and re-running -- a step that lived outside
    # the code and broke end-to-end reproducibility.
    datasets = {
        "norm": {
            "log":    str(OUTPUT_DIR / 'norm/metadata/louvain/training_log.txt'),
            "edges":  str(OUTPUT_DIR / 'norm/metadata/edge_list.txt'),
            "output": str(OUTPUT_DIR / 'norm/metadata/louvain/clustering_results.csv'),
        },
        "pred": {
            "log":    str(OUTPUT_DIR / 'pred/metadata/louvain/training_log.txt'),
            "edges":  str(OUTPUT_DIR / 'pred/metadata/edge_list.txt'),
            "output": str(OUTPUT_DIR / 'pred/metadata/louvain/clustering_results.csv'),
        },
    }

    resolutions = [0.8, 1.0, 1.8, 2.0, 5.0]

    for name, paths in datasets.items():
        # ensure output directories exist (the louvain folder may not exist yet)
        os.makedirs(os.path.dirname(paths["log"]), exist_ok=True)
        os.makedirs(os.path.dirname(paths["output"]), exist_ok=True)

        logger = Logger(paths["log"])

        # reset seeds per dataset so each dataset's run is independent yet fixed
        random.seed(SEED)
        np.random.seed(SEED)

        logger.log(f"==== Dataset: {name} ====")
        edges = load_citation_edges(paths["edges"])
        G = build_graph(edges)

        output_csv = paths["output"]
        with open(output_csv, 'w', newline='') as csvfile:
            csvwriter = csv.writer(csvfile)
            csvwriter.writerow(["Louvain Clustering Results"])
            csvwriter.writerow([])

        for resolution in resolutions:
            logger.log(f"Running Louvain clustering with resolution: {resolution}")

            # reproducible Louvain partition
            partition = perform_louvain_clustering(G, resolution=resolution, seed=SEED)

            community_dict = analyze_communities(partition)
            logger.log(f"Total communities detected: {len(community_dict)}")

            modularity = calculate_modularity_score(G, partition)
            logger.log(f"Modularity for resolution {resolution}: {modularity}")

            save_results_to_csv(resolution, community_dict, modularity, output_csv)
            logger.log("=" * 60)


if REGENERATE_COMMUNITIES and __name__ == "__main__":
    main()




<h1>Step 1b: Materialize per-resolution community files</h1>

Extracts each required resolution block from `clustering_results.csv` into clean `best_louvain_reso_*.csv` files that Steps 2–4 read. This makes the pipeline fully reproducible end-to-end (no manual copying). Community IDs and ordering are copied verbatim. Run once after Step 1.

In [ ]:
if REGENERATE_COMMUNITIES:
    # =============================================================
    # Step 1b: Materialize per-resolution community files (reproducible)
    #
    # Step 1 writes clustering_results.csv, which stacks ALL resolutions in one
    # multi-section file. The downstream steps (2, 3, 4) each read a CLEAN
    # single-resolution file named best_louvain_reso_<r>.csv. Previously those
    # clean files were produced by hand, which broke reproducibility: a fresh run
    # regenerated clustering_results.csv but not the per-resolution files.
    #
    # This cell closes that gap: it deterministically extracts each required
    # resolution block from clustering_results.csv and writes the exact
    # best_louvain_reso_*.csv files the rest of the notebook consumes. Community
    # IDs and ordering are copied verbatim (no re-sorting, no re-numbering), so the
    # output is identical to the original hand-made files.
    #
    # Requires Step 1 to have been run (clustering_results.csv present for both
    # datasets). Run this once after Step 1; Steps 2-4 then read its outputs.
    # =============================================================
    import csv
    import os
    import pandas as pd


    def extract_resolution_csv(clustering_results_path, target_resolution, output_path):
        """Extract one resolution's block from Step 1's multi-section
        clustering_results.csv and write it as a clean CSV with header
        [Community ID, Number of Nodes, Nodes]. IDs/order copied verbatim."""
        target_tag = f"Resolution: {target_resolution}"
        rows, capturing, found = [], False, False
        with open(clustering_results_path, encoding="utf-8") as f:
            for row in csv.reader(f):
                if row and row[0].startswith("Resolution:"):
                    capturing = (row[0].strip() == target_tag)
                    found = found or capturing
                    continue
                if not capturing:
                    continue
                if not row or all(c.strip() == "" for c in row):
                    capturing = False           # blank line ends the section
                    continue
                if row[0] == "Community ID":     # skip per-section header
                    continue
                rows.append(row[:3])
        if not found:
            raise ValueError(
                f"Resolution {target_resolution} not found in {clustering_results_path}. "
                "Re-run Step 1 so this resolution is present."
            )
        df = pd.DataFrame(rows, columns=["Community ID", "Number of Nodes", "Nodes"])
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        df.to_csv(output_path, index=False)
        print(f"  reso {target_resolution:>3} -> {output_path}  ({len(df)} communities)")
        return df


    # (clustering_results.csv, resolution, output filename) for every file the
    # downstream steps read. Normal uses 1.0; Predatory uses 1.8 for the main
    # analysis and 1.0 for the common-resolution sensitivity analysis.
    _EXTRACTION_PLAN = [
        (str(OUTPUT_DIR / 'norm/metadata/louvain/clustering_results.csv'), 1.0,
         str(OUTPUT_DIR / 'norm/metadata/louvain/best_louvain_reso_1.csv')),
        (str(OUTPUT_DIR / 'pred/metadata/louvain/clustering_results.csv'), 1.8,
         str(OUTPUT_DIR / 'pred/metadata/louvain/best_louvain_reso_1.8.csv')),
        (str(OUTPUT_DIR / 'pred/metadata/louvain/clustering_results.csv'), 1.0,
         str(OUTPUT_DIR / 'pred/metadata/louvain/best_louvain_reso_1.csv')),
    ]

    print("Materializing per-resolution community files from clustering_results.csv:")
    for _src, _reso, _out in _EXTRACTION_PLAN:
        extract_resolution_csv(_src, _reso, _out)
    print("Done. Downstream steps (2-4) now read reproducibly generated files.")

<h1>Check Resolution</h1>

Resolution audit for Louvain community detection.
For each dataset (Normal / Predatory) and each candidate resolution, this
script reports:
  - modularity (the quantity Louvain optimizes)
  - number of communities
  - size of the largest / median community
  - how many communities are "usable" for downstream matching (>= MIN_N nodes)

In [ ]:
if REGENERATE_COMMUNITIES:
    from community import community_louvain
    import networkx as nx
    from collections import defaultdict
    import numpy as np
    import random

    LOUVAIN_RANDOM_SEED = 42
    RESOLUTIONS = [0.8, 1.0, 1.8, 2.0, 5.0]
    MIN_N = 300  # same threshold Step 3 (cell 14) uses to filter usable communities

    # The two resolutions currently hard-coded in the community-matching step:
    CURRENT_CHOICE = {"norm": 1.0, "pred": 1.8}

    DATASETS = {
        "norm": str(OUTPUT_DIR / 'norm/metadata/edge_list.txt'),
        "pred": str(OUTPUT_DIR / 'pred/metadata/edge_list.txt'),
    }


    def load_citation_edges(file_path):
        edges = []
        with open(file_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    src, dst = int(parts[0]), int(parts[1])
                    edges.append((src, dst))
        return edges


    def build_graph(edges):
        G = nx.Graph()
        G.add_edges_from(edges)
        return G


    def audit_dataset(name, edge_file):
        random.seed(LOUVAIN_RANDOM_SEED)
        np.random.seed(LOUVAIN_RANDOM_SEED)

        edges = load_citation_edges(edge_file)
        G = build_graph(edges)

        print(f"\n{'='*78}")
        print(f"DATASET: {name.upper()}   (nodes={G.number_of_nodes():,}, "
              f"edges={G.number_of_edges():,})")
        print(f"{'='*78}")
        header = (f"{'reso':>5} | {'modularity':>10} | {'#comm':>6} | "
                  f"{'largest':>8} | {'median':>7} | {'usable(>={})':>12}".format(MIN_N))
        print(header)
        print("-" * len(header))

        rows = []
        for reso in RESOLUTIONS:
            partition = community_louvain.best_partition(
                G, resolution=reso, random_state=LOUVAIN_RANDOM_SEED
            )
            mod = community_louvain.modularity(partition, G)

            comm = defaultdict(list)
            for node, cid in partition.items():
                comm[cid].append(node)
            sizes = sorted((len(v) for v in comm.values()), reverse=True)
            n_comm = len(sizes)
            largest = sizes[0] if sizes else 0
            median = int(np.median(sizes)) if sizes else 0
            usable = sum(1 for s in sizes if s >= MIN_N)

            marker = "  <-- CURRENT" if CURRENT_CHOICE.get(name) == reso else ""
            print(f"{reso:>5} | {mod:>10.4f} | {n_comm:>6} | {largest:>8,} | "
                  f"{median:>7,} | {usable:>12}{marker}")
            rows.append({
                "reso": reso, "modularity": mod, "n_comm": n_comm,
                "largest": largest, "median": median, "usable": usable,
            })

        # ---- interpretation helpers ----
        best_mod = max(rows, key=lambda r: r["modularity"])
        # "best usable" = resolution with most usable communities; tie-break by modularity
        best_usable = max(rows, key=lambda r: (r["usable"], r["modularity"]))
        cur = next((r for r in rows if r["reso"] == CURRENT_CHOICE.get(name)), None)

        print()
        print(f"  highest modularity : reso={best_mod['reso']} "
              f"(mod={best_mod['modularity']:.4f}, usable={best_mod['usable']})")
        print(f"  most usable comms  : reso={best_usable['reso']} "
              f"(usable={best_usable['usable']}, mod={best_usable['modularity']:.4f})")
        if cur is not None:
            print(f"  CURRENT choice     : reso={cur['reso']} "
                  f"(mod={cur['modularity']:.4f}, usable={cur['usable']})")
        return rows


    def main():
        all_rows = {}
        for name, edge_file in DATASETS.items():
            all_rows[name] = audit_dataset(name, edge_file)

        print(f"\n{'='*78}")
        print("HOW TO READ THIS")
        print(f"{'='*78}")
        print(
            "- modularity alone will usually favor the LOWEST resolution (fewer,\n"
            "  bigger communities). That does NOT make it the right choice.\n"
            "- Step 3 needs several communities with >= {} nodes to match Norm vs\n"
            "  Pred. A resolution with usable=0 or 1 breaks that step.\n"
            "- A defensible choice: the resolution with a reasonable modularity\n"
            "  that ALSO yields enough usable communities for matching.\n"
            "- If the CURRENT choice (norm=1.0 / pred=1.8) still gives a sensible\n"
            "  modularity and enough usable communities, it stands. If a different\n"
            "  resolution is clearly better on BOTH, consider switching -- but then\n"
            "  re-run Step 3, since the selected community pair may change."
            .format(MIN_N)
        )


    if __name__ == "__main__":
        main()


<h1>Step 2: Draw citation network</h1>

In [ ]:

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import matplotlib.colors as mcolors
import os
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Super Node Aggregation 
def new_create_super_nodes(G, merge_threshold=0.8):
    """
    Nodes with an out-degree > 1 are merged into a repeated-target group when
    their observed cited-target sets have a Jaccard similarity >= merge_threshold
    (default 0.8) AND share at least two targets.
    Nodes with an out-degree == 1 are still grouped as identical. Nodes without references are added individually.
    """
    # Used to store nodes with out-degree == 1, grouped by exact reference sets.
    pattern_map = {}
    # Used to store nodes with out-degree > 1, each node as a candidate group.
    candidate_groups = []
    # Nodes without references.
    single_nodes = []

    for node in G.nodes():
        out_neighbors = set(G[node])
        if not out_neighbors:
            single_nodes.append(node)
        else:
            if len(out_neighbors) == 1:
                key = frozenset(out_neighbors)
                pattern_map.setdefault(key, []).append(node)
            else:
                candidate_groups.append({"pattern": out_neighbors, "nodes": [node]})

    # Non-merged group: nodes with out-degree == 1
    non_merge_groups = []
    for pattern, nodes in pattern_map.items():
        non_merge_groups.append({"pattern": set(pattern), "nodes": nodes})

    # All candidate groups with out-degree > 1 are merged using a threshold of 80% (regardless of whether there is only a single node in the group initially)
    n = len(candidate_groups)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    for i in range(n):
        for j in range(i + 1, n):
            pattern_i = candidate_groups[i]["pattern"]
            pattern_j = candidate_groups[j]["pattern"]
            intersection = pattern_i & pattern_j
            union_set = pattern_i | pattern_j
            if union_set:
                jaccard = len(intersection) / len(union_set)
                # Merge only when the shared cited-target set is a genuine
                # repeated-target pattern: Jaccard >= threshold AND at least
                # two shared targets. This matches the definition used in the
                # quantitative analysis.
                if jaccard >= merge_threshold and len(intersection) >= 2:
                    union(i, j)

    clusters = {}
    for i in range(n):
        root = find(i)
        clusters.setdefault(root, []).append(candidate_groups[i])

    merged_candidate_groups = []
    for cluster in clusters.values():
        if len(cluster) == 1:
            merged_candidate_groups.append(cluster[0])
        else:
            merged_pattern = set()
            merged_nodes = []
            for group in cluster:
                merged_pattern |= group["pattern"]
                merged_nodes.extend(group["nodes"])
            merged_candidate_groups.append({"pattern": merged_pattern, "nodes": merged_nodes})

    final_groups = non_merge_groups + merged_candidate_groups

    super_graph = nx.DiGraph()
    node_map = {}
    idx = 0
    for group in final_groups:
        super_node = f"Super_{idx}"
        idx += 1
        node_map[super_node] = group["nodes"]
        super_graph.add_node(super_node, size=len(group["nodes"]))
        for target in group["pattern"]:
            super_graph.add_edge(super_node, target)

    for n in single_nodes:
        super_graph.add_node(n, size=1)
        for target in G[n]:
            super_graph.add_edge(n, target)

    return super_graph, node_map

def filter_edge_list(edge_list_file, nodes_in_community):
    filtered_edges = []
    with open(edge_list_file, 'r') as f:
        for line in f:
            src, dst = map(int, line.strip().split())
            if src in nodes_in_community and dst in nodes_in_community:
                filtered_edges.append((src, dst))
    return filtered_edges

def visualize_5_rows_with_custom_colors(edge_list, community_data, community_id, rank, save_folder,
                                        out_deg_max=100, in_deg_max=100,
                                        blue_colors=["#c8edf7", "#535fad", "#a3245d"],
                                        orange_colors=["#fff9a0", "#3eb562", "#d95f0e"]):
    os.makedirs(save_folder, exist_ok=True)  # ensure output dir exists
   
    row = community_data[community_data['Community ID'] == community_id]
    if row.empty:
        print(f"Community {community_id} not found.")
        return

    node_list = list(map(int, row.iloc[0]['Nodes'].split(',')))

    G = nx.DiGraph()
    sub_edges = [(s, t) for s, t in edge_list if s in node_list and t in node_list]
    G.add_edges_from(sub_edges)

    super_graph, node_map = new_create_super_nodes(G)

    def is_real_super(n):
        return isinstance(n, str) and n.startswith("Super_") and super_graph.nodes[n].get('size', 1) > 1

    target_nodes = [n for n in super_graph.nodes() if super_graph.out_degree(n) == 0]
    blue_nodes   = [n for n in super_graph.nodes() if super_graph.out_degree(n) > 0]

    row1_nodes = [n for n in blue_nodes if super_graph.out_degree(n) == 1 and not is_real_super(n)]
    row2_nodes = [n for n in blue_nodes if super_graph.out_degree(n) == 1 and is_real_super(n)]
    row3_nodes = [n for n in blue_nodes if super_graph.out_degree(n) > 1 and not is_real_super(n)]
    row4_nodes = [n for n in blue_nodes if super_graph.out_degree(n) > 1 and is_real_super(n)]

    row2_nodes = sorted(row2_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
    row4_nodes = sorted(row4_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))

    
    # === Row4 Super Node statistics ===
    row4_supers = [n for n in row4_nodes if is_real_super(n)]
    stats_rows = []
    for sn in row4_supers:
        members = sorted(node_map.get(sn, [])) 
        stats_rows.append({
            "super_node": sn,
            "num_nodes_in_super": super_graph.nodes[sn].get('size', 1), 
            "num_out_edges": super_graph.out_degree(sn),                  
            "members": ",".join(map(str, members))                      
        })

    if stats_rows:
        df_row4_stats = pd.DataFrame(stats_rows)\
            .sort_values(["num_nodes_in_super", "num_out_edges", "super_node"],
                         ascending=[False, False, True])\
            .reset_index(drop=True)
    else:
        df_row4_stats = pd.DataFrame(columns=["super_node", "num_nodes_in_super", "num_out_edges", "members"])

    print(f"[Stats] Community {community_id} - Row4 super nodes: {len(df_row4_stats)}")
    if not df_row4_stats.empty:
        df_preview = df_row4_stats.copy()
        df_preview["members"] = df_preview["members"].str.slice(0, 200) + "..."
        print("Group-level statistics exported outside the repository.")

    stats_path = os.path.join(save_folder, f"row4_supernode_stats_com_{community_id}.csv")
    df_row4_stats.to_csv(stats_path, index=False)
    print(f"[Stats] Saved: {stats_path}")

    pos = {}

    def assign_positions(nodes, y_level):
        if not nodes:
            return
        x_coords = np.linspace(0, 2, len(nodes))
        for i, n in enumerate(nodes):
            pos[n] = (x_coords[i], y_level)

    assign_positions(row1_nodes, 0.9)
    assign_positions(row2_nodes, 0.7)
    assign_positions(row3_nodes, 0.55)
    assign_positions(row4_nodes, 0.4)

    target_nodes_sorted = sorted(target_nodes, key=lambda n: super_graph.in_degree(n))
    assign_positions(target_nodes_sorted, 0.0)

    GAMMA = 0.5

    cmap_out = mcolors.LinearSegmentedColormap.from_list("blue_grad", blue_colors)
    cmap_in  = mcolors.LinearSegmentedColormap.from_list("orange_grad", orange_colors)

    norm_out = mcolors.PowerNorm(gamma=GAMMA, vmin=0, vmax=out_deg_max)
    norm_in  = mcolors.PowerNorm(gamma=GAMMA, vmin=0, vmax=in_deg_max)

    def get_blue_color(n):
        return cmap_out(norm_out(super_graph.out_degree(n)))

    def get_orange_color(n):
        return cmap_in(norm_in(super_graph.in_degree(n)))

    def get_node_size(n):
        return super_graph.nodes[n].get('size', 1) * 100

    row1_colors = [get_blue_color(n) for n in row1_nodes]
    row2_colors = [get_blue_color(n) for n in row2_nodes]
    row3_colors = [get_blue_color(n) for n in row3_nodes]
    row4_colors = [get_blue_color(n) for n in row4_nodes]
    row5_colors = [get_orange_color(n) for n in target_nodes_sorted]

    row1_sizes = [get_node_size(n) for n in row1_nodes]
    row2_sizes = [get_node_size(n) for n in row2_nodes]
    row3_sizes = [get_node_size(n) for n in row3_nodes]
    row4_sizes = [get_node_size(n) for n in row4_nodes]
    row5_sizes = [get_node_size(n) for n in target_nodes_sorted]

    edges_to_draw = [(u, v) for (u, v) in super_graph.edges() if u in row4_nodes]

    edge_cmap = mpl.colors.LinearSegmentedColormap.from_list("custom_grey", ["#aebdbb", "#5e666e"])
    edge_colors = [edge_cmap(norm_out(super_graph.out_degree(u))) for (u, v) in edges_to_draw]
    # edge_colors = "#82878c"

    fig, ax = plt.subplots(figsize=(18, 12))

    ax.axis("off")

    nx.draw_networkx_nodes(super_graph, pos, nodelist=row1_nodes, node_color=row1_colors, node_size=row1_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=row2_nodes, node_color=row2_colors, node_size=row2_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=row3_nodes, node_color=row3_colors, node_size=row3_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=row4_nodes, node_color=row4_colors, node_size=row4_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=target_nodes_sorted, node_color=row5_colors, node_size=row5_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')

    nx.draw_networkx_edges(super_graph, pos, edgelist=edges_to_draw, edge_color=edge_colors, width=1,
                           alpha=0.8, ax=ax)

    # if row2_nodes:
    #     largest_row2 = max(row2_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
    #     ax.annotate(f"Size: {super_graph.nodes[largest_row2].get('size', 1)}", 
    #                 xy=pos[largest_row2], 
    #                 xytext=(pos[largest_row2][0], pos[largest_row2][1] + 0.1),
    #                 arrowprops=dict(arrowstyle="->", color="black"),
    #                 fontsize=12, 
    #                 ha='center')

    # if row4_nodes:
    #     largest_row4 = max(row4_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
    #     ax.annotate(f"Size: {super_graph.nodes[largest_row4].get('size', 1)}", 
    #                 xy=pos[largest_row4], 
    #                 xytext=(pos[largest_row4][0], pos[largest_row4][1] + 0.1),
    #                 arrowprops=dict(arrowstyle="->", color="black"),
    #                 fontsize=12, 
    #                 ha='center')


    if row2_nodes:
        largest_row2 = max(row2_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
        ax.text(pos[largest_row2][0], pos[largest_row2][1],
                f"{super_graph.nodes[largest_row2].get('size', 1)}",
                fontsize=12, color='black', ha='center', va='center')

    if row4_nodes:
        largest_row4 = max(row4_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
        ax.text(pos[largest_row4][0], pos[largest_row4][1],
                f"{super_graph.nodes[largest_row4].get('size', 1)}",
                fontsize=12, color='black', ha='center', va='center')


    label_x = -0.45
    ax.text(-0.5, 0.95, "Normal Citation Behaviors", ha='left', va='center', fontsize=15, fontweight='bold')

    ax.text(label_x, 0.9, "Row 1: Single Citation", ha='left', va='center', fontsize=13)
    ax.text(label_x, 0.7, "Row 2: Popular Paper", ha='left', va='center', fontsize=13)
    ax.text(label_x, 0.55, "Row 3: Reference List", ha='left', va='center', fontsize=13)

    ax.text(-0.5, 0.45, "Suspicious Citation Behaviors", ha='left', va='center', fontsize=15, fontweight='bold')
    ax.text(label_x, 0.4, "Row 4: Group Citation", ha='left', va='center', fontsize=13)

    ax.text(label_x, 0.0, "Row 5: Cited Papers", ha='left', va='center', fontsize=13)

    ax.text(0, -0.1, "Arrows indicate citation: citing → cited", ha='left', va='center', fontsize=13, style='italic')


    divider = make_axes_locatable(ax)
    cax_out = divider.append_axes("right", size="2%", pad=0.2)
    sm_out = mpl.cm.ScalarMappable(norm=norm_out, cmap=cmap_out)
    cbar_out = plt.colorbar(sm_out, cax=cax_out)
    cbar_out.set_label('Out-degree (Citing)', fontsize=10)

    cax_in = divider.append_axes("right", size="2%", pad=0.6)
    sm_in = mpl.cm.ScalarMappable(norm=norm_in, cmap=cmap_in)
    cbar_in = plt.colorbar(sm_in, cax=cax_in)
    cbar_in.set_label('In-degree (Cited)', fontsize=10)

    all_x = [p[0] for p in pos.values()]
    xmin = min(all_x)
    xmax = max(all_x)
    ax.set_xlim(xmin - 0.1, xmax + 0.4)
    ax.set_ylim(-0.12, 0.92)

    output_path = os.path.join(save_folder, f"TOP_{rank}_com_{community_id}.png")

    plt.savefig(output_path, format="png", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.show()
    plt.close()





Citation Network for Normal Set

In [ ]:
def main():
    #Norm Data
    csv_file = str(OUTPUT_DIR / 'norm/metadata/louvain/best_louvain_reso_1.csv') 
    edge_list_file = str(OUTPUT_DIR / 'norm/metadata/edge_list.txt') 
    save_folder = str(OUTPUT_DIR / 'norm/metadata/louvain/network_diagram')
   
    community_data = pd.read_csv(csv_file)
    top_communities = community_data.nlargest(5, 'Number of Nodes').reset_index(drop=True)

    for rank, row in top_communities.iterrows():
        target_community_id = row['Community ID']
        all_nodes = set(map(int, row['Nodes'].split(',')))
        filtered_edges = filter_edge_list(edge_list_file, all_nodes)

        visualize_5_rows_with_custom_colors(
            filtered_edges,
            community_data,
            target_community_id,
            rank + 1, 
            save_folder,
            out_deg_max=100,
            in_deg_max=100,
            blue_colors=["#c8edf7", "#535fad", "#a3245d"],
            orange_colors=["#fff9a0", "#3eb562", "#d95f0e"]
        )

if __name__ == "__main__":
    main()

Citation Network for Predatory Set

In [ ]:
def main():
    # Pred Data:
    csv_file = str(OUTPUT_DIR / 'pred/metadata/louvain/best_louvain_reso_1.8.csv') 
    edge_list_file = str(OUTPUT_DIR / 'pred/metadata/edge_list.txt') 
    save_folder = str(OUTPUT_DIR / 'pred/metadata/louvain/network_diagram')

    community_data = pd.read_csv(csv_file)
    top_communities = community_data.nlargest(5, 'Number of Nodes').reset_index(drop=True)

    for rank, row in top_communities.iterrows():
        target_community_id = row['Community ID']
        all_nodes = set(map(int, row['Nodes'].split(',')))
        filtered_edges = filter_edge_list(edge_list_file, all_nodes)

        visualize_5_rows_with_custom_colors(
            filtered_edges,
            community_data,
            target_community_id,
            rank + 1, 
            save_folder,
            out_deg_max=100,
            in_deg_max=100,
            blue_colors=["#c8edf7", "#535fad", "#a3245d"],
            orange_colors=["#fff9a0", "#3eb562", "#d95f0e"]
        )

if __name__ == "__main__":
    main()

Draw citation network for a specific community

In [ ]:
if DRAW_SELECTED_COMMUNITY:
    DATASET = "pred"     # "norm" 或 "pred"
    COMMUNITY_ID = None  

    def main():
        if DATASET == "norm":
            csv_file = str(OUTPUT_DIR / 'norm/metadata/louvain/best_louvain_reso_1.csv')
            edge_list_file = str(OUTPUT_DIR / 'norm/metadata/edge_list.txt')
            save_folder = str(OUTPUT_DIR / 'norm/metadata/louvain/network_diagram')
        elif DATASET == "pred":
            csv_file = str(OUTPUT_DIR / 'pred/metadata/louvain/best_louvain_reso_1.8.csv')
            edge_list_file = str(OUTPUT_DIR / 'pred/metadata/edge_list.txt')
            save_folder = str(OUTPUT_DIR / 'pred/metadata/louvain/network_diagram')
        else:
            raise ValueError("DATASET can only be 'norm' or 'pred'")

        if COMMUNITY_ID is None:
            raise ValueError("Set COMMUNITY_ID。")

        os.makedirs(save_folder, exist_ok=True)

        community_data = pd.read_csv(csv_file)

        row = community_data[community_data['Community ID'] == int(COMMUNITY_ID)]
        if row.empty:
            raise ValueError(f"Can't find Community ID = {COMMUNITY_ID}（in {DATASET}")

        all_nodes = set(map(int, str(row.iloc[0]['Nodes']).split(',')))
        filtered_edges = filter_edge_list(edge_list_file, all_nodes)

        visualize_5_rows_with_custom_colors(
            filtered_edges,
            community_data,
            int(COMMUNITY_ID),
            1,
            save_folder,
            out_deg_max=100,
            in_deg_max=100,
            blue_colors=["#c8edf7", "#535fad", "#a3245d"],
            orange_colors=["#fff9a0", "#3eb562", "#d95f0e"]
        )

    if __name__ == "__main__":
        main()



<h1>Step 3: Find the most similar communities between Pred and Norm Sets to compare</h1>

In [ ]:
import pandas as pd

def load_communities(csv_file):
    df = pd.read_csv(csv_file).rename(columns={'Community ID':'CommunityID','Number of Nodes':'NumNodes'})
    if not {'CommunityID','NumNodes','Nodes'}.issubset(df.columns):
        raise ValueError('Community CSV schema is incomplete.')
    communities = {}
    for _, row in df.iterrows():
        comm_id = str(row["CommunityID"]).strip()
        nodes_str = str(row["Nodes"]).strip().strip('"').replace("[","").replace("]","")
        nodes = set(n.strip() for n in nodes_str.split(",") if n.strip())
        communities[comm_id] = nodes
    return communities

def load_edge_list(edge_file):
    edges = []
    with open(edge_file, 'r', encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                source, target = str(parts[0]).strip(), str(parts[1]).strip()
                edges.append((source, target))
    return edges

def compute_community_metrics(community_nodes, edges):
    community_edges = [(s, t) for s, t in edges if s in community_nodes and t in community_nodes]

    total_nodes = len(community_nodes)
    total_edges = len(community_edges)

    citing_nodes = set(s for s, t in community_edges)
    citing_count = len(citing_nodes)

    cited_nodes = set(t for s, t in community_edges)
    cited_count = len(cited_nodes)

    prop_citing = (citing_count / total_nodes) if total_nodes > 0 else 0.0
    prop_cited  = (cited_count  / total_nodes) if total_nodes > 0 else 0.0

    return {
        "total_nodes": total_nodes,
        "total_edges": total_edges,
        "citing_count": citing_count,
        "cited_count": cited_count,
        "prop_citing": prop_citing,
        "prop_cited":  prop_cited
    }

def similarity_component(a, b):
    s = a + b
    if s == 0:
        return 1.0
    v = 1.0 - abs(a - b) / s
    return max(0.0, min(1.0, v))

def compute_similarity(m1, m2):
    """
    Core: Compares the two sides' internal communities.
    - prop_citing: The percentage of nodes that "issue citations" within the community.
    - prop_cited: The percentage of nodes that "be cited" within the community.
    default overall = (score_prop_citing + score_prop_cited) / 2
    """
    score_prop_citing = similarity_component(m1["prop_citing"], m2["prop_citing"])
    score_prop_cited  = similarity_component(m1["prop_cited"],  m2["prop_cited"])

    # score_nodes = similarity_component(m1["total_nodes"], m2["total_nodes"])
    # score_edges = similarity_component(m1["total_edges"], m2["total_edges"]) if (m1["total_edges"] + m2["total_edges"]) > 0 else 1.0
    # overall = (0.4*(score_prop_citing + score_prop_cited) + 0.1*score_nodes + 0.1*score_edges)

    overall = (score_prop_citing + score_prop_cited) / 2.0
    details = {
        "score_prop_citing": score_prop_citing,
        "score_prop_cited":  score_prop_cited,
        # "score_nodes": score_nodes,
        # "score_edges": score_edges,
    }
    return overall, details

def main():
    good_comm_file = str(OUTPUT_DIR / 'norm/metadata/louvain/best_louvain_reso_1.csv')
    bad_comm_file  = str(OUTPUT_DIR / 'pred/metadata/louvain/best_louvain_reso_1.8.csv')
    good_edge_file = str(OUTPUT_DIR / 'norm/metadata/edge_list.txt')
    bad_edge_file  = str(OUTPUT_DIR / 'pred/metadata/edge_list.txt')

    good_communities = load_communities(good_comm_file)
    bad_communities  = load_communities(bad_comm_file)
    good_edges = load_edge_list(good_edge_file)
    bad_edges  = load_edge_list(bad_edge_file)

    good_metrics = {cid: compute_community_metrics(nodes, good_edges) for cid, nodes in good_communities.items()}
    bad_metrics  = {cid: compute_community_metrics(nodes, bad_edges)  for cid, nodes in bad_communities.items()}

    MIN_N = 300
    good_filtered = {cid:m for cid,m in good_metrics.items() if m["total_nodes"] >= MIN_N}
    bad_filtered  = {cid:m for cid,m in bad_metrics.items()  if m["total_nodes"] >= MIN_N}

    if not good_filtered or not bad_filtered:
        raise ValueError("Not such community after filter")

    best_score = -1.0
    best_pair = None
    best_detail = None

    for g_comm, g_met in good_filtered.items():
        for b_comm, b_met in bad_filtered.items():
            sim_score, detail_scores = compute_similarity(g_met, b_met)
            if sim_score > best_score:
                best_score = sim_score
                best_pair = (g_comm, b_comm)
                best_detail = detail_scores

    print("\nMost similar community pair (by proportions):")
    print("Norm community ID:", best_pair[0], "Metrics:", good_filtered[best_pair[0]])
    print("Pred community ID:", best_pair[1], "Metrics:", bad_filtered[best_pair[1]])
    print("Final similarity score:", round(best_score, 6))
    print("Detail (prop-based):", best_detail)

if __name__ == '__main__':
    main()


<h1> Step 4: Group-citation performance in overall dataste</h1>

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from datetime import datetime
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu


BASE = OUTPUT_DIR
RUN_DIR = OUTPUT_DIR / f"citation_amplification_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
JACCARD_THRESHOLD = 0.80
MIN_SHARED_TARGETS = 2
PERMUTATIONS = 20_000
RANDOM_SEED = 42

SOURCES = {
    "Predatory": (
        OUTPUT_DIR / "pred/metadata/louvain/best_louvain_reso_1.8.csv",
        OUTPUT_DIR / "pred/metadata/edge_list.txt",
    ),
    "Normal": (
        OUTPUT_DIR / "norm/metadata/louvain/best_louvain_reso_1.csv",
        OUTPUT_DIR / "norm/metadata/edge_list.txt",
    ),
}


def parse_nodes(value: str) -> set[int]:
    return {int(x.strip()) for x in str(value).split(",") if x.strip()}


def load_edges(path: Path) -> set[tuple[int, int]]:
    with path.open(encoding="utf-8") as handle:
        return {tuple(map(int, line.split()[:2])) for line in handle if line.strip()}


def build_similarity_graph(target_sets: dict[int, set[int]]) -> nx.Graph:
    graph = nx.Graph()
    graph.add_nodes_from(target_sets)
    shared_counts: Counter[tuple[int, int]] = Counter()
    target_to_citers: dict[int, list[int]] = defaultdict(list)
    for citing_paper, targets in target_sets.items():
        for target in targets:
            target_to_citers[target].append(citing_paper)
    for citers in target_to_citers.values():
        for left, right in combinations(sorted(citers), 2):
            shared_counts[(left, right)] += 1
    for (left, right), shared in shared_counts.items():
        if shared < MIN_SHARED_TARGETS:
            continue
        union = len(target_sets[left] | target_sets[right])
        jaccard = shared / union
        if jaccard >= JACCARD_THRESHOLD:
            graph.add_edge(left, right, shared_targets=shared, jaccard=jaccard)
    return graph


def analyze_dataset(label: str, community_path: Path, edge_path: Path):
    communities = pd.read_csv(community_path)
    deduplicated_edges = load_edges(edge_path)
    metrics_rows, group_rows, target_rows = [], [], []
    excluded_no_targets = 0

    for _, row in communities.iterrows():
        community_id = int(row["Community ID"])
        nodes = parse_nodes(row["Nodes"])
        within_edges = {(s, t) for s, t in deduplicated_edges if s in nodes and t in nodes}
        observed_targets: dict[int, set[int]] = defaultdict(set)
        for source, target in within_edges:
            observed_targets[source].add(target)
        eligible_citers = {
            source: targets for source, targets in observed_targets.items() if len(targets) >= 2
        }
        eligible_targets = set().union(*eligible_citers.values()) if eligible_citers else set()
        if not eligible_targets:
            excluded_no_targets += 1
            continue

        similarity_graph = build_similarity_graph(eligible_citers)
        components = [
            sorted(component)
            for component in nx.connected_components(similarity_graph)
            if len(component) >= 2
        ]
        components.sort(key=lambda members: (-len(members), members))
        group_ids = {}
        for index, members in enumerate(components, 1):
            group_id = f"{label}_C{community_id}_G{index}"
            group_ids[group_id] = members
            links = []
            for left, right in similarity_graph.subgraph(members).edges():
                data = similarity_graph[left][right]
                links.append(
                    {
                        "paper_1": int(left),
                        "paper_2": int(right),
                        "jaccard": float(data["jaccard"]),
                        "shared_targets": int(data["shared_targets"]),
                    }
                )
            group_rows.append(
                {
                    "dataset": label,
                    "community_id": community_id,
                    "group_id": group_id,
                    "group_size": len(members),
                    "citing_paper_ids": json.dumps(members),
                    "observed_target_sets": json.dumps(
                        {str(p): sorted(eligible_citers[p]) for p in members}, sort_keys=True
                    ),
                    "qualifying_similarity_links": json.dumps(links, sort_keys=True),
                }
            )

        target_group_counts: dict[int, dict[str, int]] = defaultdict(dict)
        for group_id, members in group_ids.items():
            for target in eligible_targets:
                count = sum(target in eligible_citers[paper] for paper in members)
                if count:
                    target_group_counts[target][group_id] = count

        amplified_targets = {
            target
            for target, counts in target_group_counts.items()
            if any(count >= 2 for count in counts.values())
        }
        multi_group_targets = {
            target
            for target, counts in target_group_counts.items()
            if sum(count >= 1 for count in counts.values()) >= 2
        }
        amplified_citation_counts = []
        for target in sorted(amplified_targets):
            counts = target_group_counts[target]
            total_group_member_citations = sum(counts.values())
            amplified_citation_counts.append(total_group_member_citations)
            target_rows.append(
                {
                    "dataset": label,
                    "community_id": community_id,
                    "target_paper_id": target,
                    "is_group_amplified": True,
                    "is_multi_group": target in multi_group_targets,
                    "group_member_citations": total_group_member_citations,
                    "citing_group_counts": json.dumps(counts, sort_keys=True),
                }
            )

        metrics_rows.append(
            {
                "dataset": label,
                "community_id": community_id,
                "eligible_citing_papers": len(eligible_citers),
                "eligible_targets": len(eligible_targets),
                "repeated_target_groups": len(components),
                "papers_in_repeated_target_groups": sum(map(len, components)),
                "group_amplified_targets": len(amplified_targets),
                "multi_group_targets": len(multi_group_targets),
                "median_group_member_citations_per_amplified_target": (
                    float(np.median(amplified_citation_counts))
                    if amplified_citation_counts
                    else np.nan
                ),
                "amplified_target_rate": len(amplified_targets) / len(eligible_targets),
            }
        )
    return metrics_rows, group_rows, target_rows, excluded_no_targets, len(deduplicated_edges)


def distribution_summary(values: pd.Series) -> dict:
    q1, median, q3 = values.quantile([0.25, 0.50, 0.75])
    return {
        "n": int(values.size),
        "mean": float(values.mean()),
        "median": float(median),
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(q3 - q1),
    }


def compare(metrics: pd.DataFrame) -> dict:
    pred = metrics.loc[metrics.dataset == "Predatory", "amplified_target_rate"].to_numpy()
    norm = metrics.loc[metrics.dataset == "Normal", "amplified_target_rate"].to_numpy()
    u_stat, mw_p = mannwhitneyu(pred, norm, alternative="two-sided")
    rank_biserial = 2 * u_stat / (len(pred) * len(norm)) - 1
    observed = float(pred.mean() - norm.mean())
    pooled = np.concatenate([pred, norm])
    rng = np.random.default_rng(RANDOM_SEED)
    extreme = 0
    for _ in range(PERMUTATIONS):
        permuted = rng.permutation(pooled)
        difference = permuted[: len(pred)].mean() - permuted[len(pred) :].mean()
        extreme += abs(difference) >= abs(observed)
    return {
        "Predatory": distribution_summary(pd.Series(pred)),
        "Normal": distribution_summary(pd.Series(norm)),
        "mann_whitney_u": float(u_stat),
        "mann_whitney_two_sided_p": float(mw_p),
        "rank_biserial_predatory_minus_normal": float(rank_biserial),
        "mean_difference_predatory_minus_normal": observed,
        "permutation_test": {
            "permutations": PERMUTATIONS,
            "seed": RANDOM_SEED,
            "two_sided_p": (extreme + 1) / (PERMUTATIONS + 1),
        },
    }


def dataset_description(label: str, metrics: pd.DataFrame, groups: pd.DataFrame) -> dict:
    subset = metrics[metrics.dataset == label]
    group_sizes = groups.loc[groups.dataset == label, "group_size"]
    eligible = int(subset.eligible_targets.sum())
    amplified = int(subset.group_amplified_targets.sum())
    multi = int(subset.multi_group_targets.sum())
    return {
        "total_eligible_targets": eligible,
        "total_group_amplified_targets": amplified,
        "descriptive_amplified_target_percentage": 100 * amplified / eligible,
        "total_multi_group_targets": multi,
        "descriptive_multi_group_target_percentage": 100 * multi / eligible,
        "total_repeated_target_citing_groups": int(len(group_sizes)),
        "group_size_distribution": {
            "counts": {str(k): int(v) for k, v in group_sizes.value_counts().sort_index().items()},
            "min": int(group_sizes.min()) if len(group_sizes) else None,
            "median": float(group_sizes.median()) if len(group_sizes) else None,
            "max": int(group_sizes.max()) if len(group_sizes) else None,
        },
    }


def validation_examples(groups: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    rows = []
    for label in ("Predatory", "Normal"):
        candidates = groups[groups.dataset == label]
        chosen = candidates.iloc[
            rng.choice(len(candidates), size=min(5, len(candidates)), replace=False)
        ]
        for _, group in chosen.iterrows():
            members = json.loads(group.citing_paper_ids)
            target_sets = {
                int(k): set(v) for k, v in json.loads(group.observed_target_sets).items()
            }
            pairwise = []
            for left, right in combinations(members, 2):
                shared = len(target_sets[left] & target_sets[right])
                union = len(target_sets[left] | target_sets[right])
                pairwise.append(
                    {
                        "paper_1": left,
                        "paper_2": right,
                        "jaccard": shared / union,
                        "shared_targets": shared,
                        "qualifying_link": shared >= 2 and shared / union >= 0.80,
                    }
                )
            links = json.loads(group.qualifying_similarity_links)
            assert links
            assert all(
                link["jaccard"] >= JACCARD_THRESHOLD
                and link["shared_targets"] >= MIN_SHARED_TARGETS
                for link in links
            )
            community_targets = targets.loc[
                (targets.dataset == label) & (targets.community_id == group.community_id)
            ]
            amplified_ids = [
                int(target.target_paper_id)
                for _, target in community_targets.iterrows()
                if json.loads(target.citing_group_counts).get(group.group_id, 0) >= 2
            ]
            rows.append(
                {
                    "dataset": label,
                    "community_id": int(group.community_id),
                    "group_id": group.group_id,
                    "citing_paper_ids": json.dumps(members),
                    "observed_target_sets": group.observed_target_sets,
                    "pairwise_comparisons": json.dumps(pairwise, sort_keys=True),
                    "qualifying_similarity_links": group.qualifying_similarity_links,
                    "amplified_target_ids": json.dumps(sorted(amplified_ids)),
                    "manual_rule_check": "PASS: connected by links with Jaccard >= 0.80 and >= 2 shared targets",
                }
            )
    return pd.DataFrame(rows)


def make_figures(metrics: pd.DataFrame) -> None:
    colors = {"Predatory": "#a3245d", "Normal": "#535fad"}
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    values = [
        metrics.loc[metrics.dataset == label, "amplified_target_rate"]
        for label in ("Predatory", "Normal")
    ]
    box = ax.boxplot(values, tick_labels=["Predatory", "Normal"], patch_artist=True, showfliers=False)
    for patch, label in zip(box["boxes"], ("Predatory", "Normal")):
        patch.set_facecolor(colors[label])
        patch.set_alpha(0.65)
    ax.set_ylabel("Community amplified-target rate")
    ax.set_title("Repeated-target amplification across citation communities")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "amplified_target_rate_distribution.png", dpi=300)
    plt.close(fig)

    plotted = metrics.sort_values("amplified_target_rate", ascending=False).head(30).copy()
    plotted["label"] = plotted.dataset.str[0] + plotted.community_id.astype(str)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(
        plotted.label,
        plotted.amplified_target_rate,
        color=[colors[x] for x in plotted.dataset],
    )
    ax.set_ylabel("Amplified-target rate")
    ax.set_xlabel("Community (P = Predatory; N = Normal)")
    ax.set_title("Figure 6. Communities with highest repeated-target amplification")
    ax.tick_params(axis="x", rotation=70)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "figure6_repeated_target_amplification.png", dpi=300)
    plt.close(fig)


def main() -> Path:
    metric_rows, group_rows, target_rows = [], [], []
    excluded, edge_counts = {}, {}
    for label, paths in SOURCES.items():
        m, g, t, x, edge_count = analyze_dataset(label, *paths)
        metric_rows.extend(m)
        group_rows.extend(g)
        target_rows.extend(t)
        excluded[label] = x
        edge_counts[label] = edge_count

    metrics = pd.DataFrame(metric_rows)
    groups = pd.DataFrame(group_rows)
    targets = pd.DataFrame(target_rows)
    validation = validation_examples(groups, targets)
    sensitivity = metrics[metrics.eligible_targets >= 5]
    summary = {
        "definition": {
            "observed_target_sets_only": True,
            "jaccard_threshold": JACCARD_THRESHOLD,
            "minimum_shared_targets": MIN_SHARED_TARGETS,
            "group": "connected component with at least two eligible citing papers",
            "inferential_unit": "Louvain community",
        },
        "deduplicated_directed_edge_counts": edge_counts,
        "communities_excluded_for_no_eligible_targets": excluded,
        "primary_comparison": compare(metrics),
        "sensitivity_eligible_targets_at_least_5": compare(sensitivity),
        "descriptive_dataset_results": {
            label: dataset_description(label, metrics, groups)
            for label in ("Predatory", "Normal")
        },
        "interpretation": (
            "Targets and papers are nested within communities, so pooled dataset percentages "
            "are descriptive. This measures repeated-target amplification in the observed "
            "Google Scholar citation graph. Citation anomalies informed seed-profile selection; "
            "the result characterizes the selected ecosystems and does not independently "
            "validate the Predatory label."
        ),
        "figure_6_definition_matches_quantitative_analysis": True,
    }

    metrics.to_csv(RUN_DIR / "community_amplification_metrics.csv", index=False)
    groups.to_csv(RUN_DIR / "repeated_target_groups.csv", index=False)
    targets.to_csv(RUN_DIR / "amplified_targets.csv", index=False)
    validation.to_csv(RUN_DIR / "validation_examples.csv", index=False)
    (RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    make_figures(metrics)

    print(json.dumps(summary, indent=2))
    print("\nInternal validation (five sampled groups per dataset):")
    print("Internal numeric rule checks passed; individual paper examples are not displayed.")
    print(f"\nOutputs saved to: {RUN_DIR}")
    return RUN_DIR



print("Repeated-target amplification configuration")
print(f"Jaccard threshold: {JACCARD_THRESHOLD}")
print(f"Minimum shared targets: {MIN_SHARED_TARGETS}")
print(f"Permutation draws: {PERMUTATIONS}")
print(f"Analysis/validation seed: {RANDOM_SEED}")

citation_amplification_run_dir = main()
print(f"Completed analysis: {citation_amplification_run_dir}")


<h3>Step 4b: Sensitivity analysis — common Louvain resolution (1.0)</h3>

Re-runs the identical amplification pipeline with **both** groups at resolution 1.0 to confirm the group difference is not an artifact of using different resolutions in the main analysis. Reuses the Step 4 functions; does not alter the main analysis or its outputs. Run Step 4 first.

In [ ]:
# =============================================================
# Sensitivity analysis: COMMON Louvain resolution = 1.0
#
# Main analysis uses each group's own resolution (Predatory 1.8, Normal 1.0),
# chosen for per-group modularity. Reviewer concern: could the group difference
# be an artifact of using DIFFERENT resolutions rather than a real effect? To
# rule this out, we re-run the identical amplification pipeline with BOTH groups
# at resolution 1.0 and compare direction and magnitude.
#
# Why 1.0 (not 1.8) is the common value: at resolution 1.8 the Normal graph has
# no community >= 300 nodes, so matched-community analysis is impossible there;
# 1.0 yields usable communities for both groups.
#
# Reuses the Step 4 functions (analyze_dataset, compare) verbatim, so the
# sensitivity result comes from the same code as the main result. Reads the
# Predatory 1.0 file produced reproducibly in Step 1b. Run Step 1b and Step 4
# first. Does not modify the main analysis or its outputs.
# =============================================================
from datetime import datetime
from pathlib import Path
import json
import pandas as pd

SENS_SOURCES = {
    "Predatory": (
        OUTPUT_DIR / "pred/metadata/louvain/best_louvain_reso_1.csv",   # 1.0 (main uses 1.8)
        OUTPUT_DIR / "pred/metadata/edge_list.txt",
    ),
    "Normal": (
        OUTPUT_DIR / "norm/metadata/louvain/best_louvain_reso_1.csv",   # 1.0 (same as main)
        OUTPUT_DIR / "norm/metadata/edge_list.txt",
    ),
}

SENS_RUN_DIR = OUTPUT_DIR / f"sensitivity_common_reso_1.0_{datetime.now():%Y%m%d_%H%M%S}"
SENS_RUN_DIR.mkdir(parents=True, exist_ok=True)

# --- run the SAME pipeline, both groups at resolution 1.0 ---
sens_metric_rows = []
for label, paths in SENS_SOURCES.items():
    m, _g, _t, _x, _edge = analyze_dataset(label, *paths)
    sens_metric_rows.extend(m)

sens_metrics = pd.DataFrame(sens_metric_rows)
sens_comparison = compare(sens_metrics)

sens_metrics.to_csv(SENS_RUN_DIR / "community_amplification_metrics_common_reso.csv", index=False)
(SENS_RUN_DIR / "sensitivity_summary.json").write_text(
    json.dumps(
        {
            "common_resolution": 1.0,
            "note": "Both groups at resolution 1.0; same pipeline as the main analysis.",
            "comparison": sens_comparison,
        },
        indent=2,
    ),
    encoding="utf-8",
)


# def _one_line(tag, comp):
#     return (
#         f"{tag:>28}: Pred median={comp['Predatory']['median']:.4f}  "
#         f"Norm median={comp['Normal']['median']:.4f}  "
#         f"mean diff (P-N)={comp['mean_difference_predatory_minus_normal']:+.4f}  "
#         f"MW p={comp['mann_whitney_two_sided_p']:.3g}  "
#         f"perm p={comp['permutation_test']['two_sided_p']:.3g}  "
#         f"rank-biserial={comp['rank_biserial_predatory_minus_normal']:+.3f}"
#     )


def _one_line(tag, comp):
    return (
        f"{tag:>28}: "
        f"Pred n={comp['Predatory']['n']}  "
        f"Norm n={comp['Normal']['n']}  "
        f"Pred median={comp['Predatory']['median']:.4f}  "
        f"Norm median={comp['Normal']['median']:.4f}  "
        f"mean diff (P-N)={comp['mean_difference_predatory_minus_normal']:+.4f}  "
        f"MW p={comp['mann_whitney_two_sided_p']:.3g}  "
        f"perm p={comp['permutation_test']['two_sided_p']:.3g}  "
        f"rank-biserial={comp['rank_biserial_predatory_minus_normal']:+.3f}"
    )

print("\n=== Sensitivity: common resolution = 1.0 (both groups) ===")
print(_one_line("common reso 1.0", sens_comparison))
try:
    _main = json.loads((citation_amplification_run_dir / "summary.json").read_text(encoding="utf-8"))
    print(_one_line("main (Pred 1.8 / Norm 1.0)", _main["primary_comparison"]))
    print(
        "\nRobustness read: the conclusion is robust if the sign of the mean "
        "difference and the significance are preserved across both rows and the "
        "effect magnitude stays comparable."
    )
except Exception as e:
    print(f"(Main summary not loaded for side-by-side: {e})")

print(f"\nSensitivity outputs saved to: {SENS_RUN_DIR}")

<h3>Apply AI Ratio</h3>

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import matplotlib.colors as mcolors
import os
from mpl_toolkits.axes_grid1 import make_axes_locatable

def visualize_5_rows_with_AI_colors(edge_list, community_data, community_id, rank, save_folder,
                                        vote_results_pkl,
                                        out_deg_max=100):
    os.makedirs(save_folder, exist_ok=True)  # ensure output dir exists
  
    df_votes = pd.read_pickle(vote_results_pkl)
    ai_dict = {}
    for _, row_vote in df_votes.iterrows():
        ai_dict[row_vote['paper_id']] = {'AI':1.,'Human':0.}.get(row_vote['final_decision'],np.nan)

    row = community_data[community_data['Community ID'] == community_id]
    if row.empty:
        print(f"Community {community_id} not found.")
        return

    node_list = list(map(int, row.iloc[0]['Nodes'].split(',')))

    G = nx.DiGraph()
    sub_edges = [(s, t) for s, t in edge_list if s in node_list and t in node_list]
    G.add_edges_from(sub_edges)

    super_graph, node_map = new_create_super_nodes(G)

    for super_node, papers in node_map.items():
        available = [ai_dict.get(p,np.nan) for p in papers]
        available = [value for value in available if np.isfinite(value)]
        ai_ratio = float(np.mean(available)) if available else np.nan
        super_graph.nodes[super_node]['ai_ratio'] = ai_ratio

    for node in super_graph.nodes():
        if node not in node_map:
            super_graph.nodes[node]['ai_ratio'] = ai_dict.get(node,np.nan)

    def is_real_super(n):
        return isinstance(n, str) and n.startswith("Super_") and super_graph.nodes[n].get('size', 1) > 1

    target_nodes = [n for n in super_graph.nodes() if super_graph.out_degree(n) == 0]
    blue_nodes   = [n for n in super_graph.nodes() if super_graph.out_degree(n) > 0]

    row1_nodes = [n for n in blue_nodes if super_graph.out_degree(n) == 1 and not is_real_super(n)]
    row2_nodes = [n for n in blue_nodes if super_graph.out_degree(n) == 1 and is_real_super(n)]
    row3_nodes = [n for n in blue_nodes if super_graph.out_degree(n) > 1 and not is_real_super(n)]
    row4_nodes = [n for n in blue_nodes if super_graph.out_degree(n) > 1 and is_real_super(n)]

    row2_nodes = sorted(row2_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
    row4_nodes = sorted(row4_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))

    pos = {}

    def assign_positions(nodes, y_level):
        if not nodes:
            return
        x_coords = np.linspace(0, 2, len(nodes))
        for i, n in enumerate(nodes):
            pos[n] = (x_coords[i], y_level)

    assign_positions(row1_nodes, 0.85)
    assign_positions(row2_nodes, 0.7)
    assign_positions(row3_nodes, 0.55)
    assign_positions(row4_nodes, 0.4)

    target_nodes_sorted = sorted(target_nodes, key=lambda n: super_graph.in_degree(n))
    assign_positions(target_nodes_sorted, 0.0)

    ai_cmap = plt.get_cmap('YlOrRd')      
    norm_ai = mpl.colors.Normalize(vmin=0, vmax=1)

    def get_ai_color(n):
        value = super_graph.nodes[n].get('ai_ratio',np.nan)
        return ai_cmap(norm_ai(value)) if np.isfinite(value) else '#999999'

    row1_colors = [get_ai_color(n) for n in row1_nodes]
    row2_colors = [get_ai_color(n) for n in row2_nodes]
    row3_colors = [get_ai_color(n) for n in row3_nodes]
    row4_colors = [get_ai_color(n) for n in row4_nodes]
    row5_colors = [get_ai_color(n) for n in target_nodes_sorted]

    def get_node_size(n):
        return super_graph.nodes[n].get('size', 1) * 100

    row1_sizes = [get_node_size(n) for n in row1_nodes]
    row2_sizes = [get_node_size(n) for n in row2_nodes]
    row3_sizes = [get_node_size(n) for n in row3_nodes]
    row4_sizes = [get_node_size(n) for n in row4_nodes]
    row5_sizes = [get_node_size(n) for n in target_nodes_sorted]

    edges_to_draw = [(u, v) for (u, v) in super_graph.edges() if u in row4_nodes]

    edge_cmap = mpl.colors.LinearSegmentedColormap.from_list("custom_grey", ["#aebdbb", "#5e666e"])
    edge_colors = [edge_cmap(mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=out_deg_max)(super_graph.out_degree(u)))
                   for (u, v) in edges_to_draw]

    fig, ax = plt.subplots(figsize=(20, 11))

    ax.axis("off")

    nx.draw_networkx_nodes(super_graph, pos, nodelist=row1_nodes, node_color=row1_colors, node_size=row1_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=row2_nodes, node_color=row2_colors, node_size=row2_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=row3_nodes, node_color=row3_colors, node_size=row3_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=row4_nodes, node_color=row4_colors, node_size=row4_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')
    nx.draw_networkx_nodes(super_graph, pos, nodelist=target_nodes_sorted, node_color=row5_colors, node_size=row5_sizes,
                           alpha=1.0, ax=ax, linewidths=0.8, edgecolors='grey')

    if row2_nodes:
        largest_row2 = max(row2_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
        ax.text(pos[largest_row2][0], pos[largest_row2][1],
                f"{super_graph.nodes[largest_row2].get('size', 1)}",
                fontsize=12, color='black', ha='center', va='center')

    if row4_nodes:
        largest_row4 = max(row4_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
        ax.text(pos[largest_row4][0], pos[largest_row4][1],
                f"{super_graph.nodes[largest_row4].get('size', 1)}",
                fontsize=12, color='black', ha='center', va='center')


    # if row2_nodes:
    #     largest_row2 = max(row2_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
    #     ax.annotate(f"Size: {super_graph.nodes[largest_row2].get('size', 1)}", 
    #                 xy=pos[largest_row2], 
    #                 xytext=(pos[largest_row2][0], pos[largest_row2][1] + 0.112),
    #                 arrowprops=dict(arrowstyle="->", color="black"),
    #                 fontsize=12, 
    #                 ha='center')

    # if row4_nodes:
    #     largest_row4 = max(row4_nodes, key=lambda n: super_graph.nodes[n].get('size', 1))
    #     ax.annotate(f"Size: {super_graph.nodes[largest_row4].get('size', 1)}", 
    #                 xy=pos[largest_row4], 
    #                 xytext=(pos[largest_row4][0], pos[largest_row4][1] + 0.1),
    #                 arrowprops=dict(arrowstyle="->", color="black"),
    #                 fontsize=12, 
    #                 ha='center')

    nx.draw_networkx_edges(super_graph, pos, edgelist=edges_to_draw, edge_color=edge_colors, width=1,
                           alpha=0.8, ax=ax)

    label_x = -0.5
    ax.text(-0.5, 0.95, "Normal Citation Behaviors", ha='left', va='center', fontsize=15, fontweight='bold')

    ax.text(label_x, 0.85, "Row 1: Single Citation", ha='left', va='center', fontsize=13)
    ax.text(label_x, 0.7, "Row 2: Popular Paper", ha='left', va='center', fontsize=13)
    ax.text(label_x, 0.55, "Row 3: Reference List", ha='left', va='center', fontsize=13)

    ax.text(-0.5, 0.45, "Suspicious Citation Behaviors", ha='left', va='center', fontsize=15, fontweight='bold')
    ax.text(label_x, 0.4, "Row 4: Group Citation", ha='left', va='center', fontsize=13)

    ax.text(label_x, 0.0, "Row 5: Cited Papers", ha='left', va='center', fontsize=13)

    ax.text(0, -0.1, "Arrows indicate citation: citing → cited", ha='left', va='center', fontsize=13, style='italic')

    divider = make_axes_locatable(ax)
    cax_ai = divider.append_axes("right", size="2%", pad=0.5)
    sm_ai = mpl.cm.ScalarMappable(norm=norm_ai, cmap=ai_cmap)
    cbar_ai = plt.colorbar(sm_ai, cax=cax_ai)
    cbar_ai.set_label('AI ratio', fontsize=10)

    ax.set_ylim(-0.10, 0.90) 
    all_x = [p[0] for p in pos.values()]  
    xmin = min(all_x)
    xmax = max(all_x)
    ax.set_xlim(xmin - 0.6, xmax + 0.4)

    output_path = os.path.join(save_folder, f"AI_ratio_{rank}_com_{community_id}.png")
    plt.savefig(output_path, format="png", dpi=300, bbox_inches='tight', pad_inches=0.09)
    plt.show()
    plt.close()

Norm Data

In [ ]:
def main():
    csv_file = str(OUTPUT_DIR / 'norm/metadata/louvain/best_louvain_reso_1.csv')
    edge_list_file = str(OUTPUT_DIR / 'norm/metadata/edge_list.txt')
    save_folder = str(OUTPUT_DIR / 'norm/metadata/louvain/ai_network_diagram')
    vote_results_pkl = str(DATA_DIR / 'norm/dataset/pkl/vote_results.pkl')

    community_data = pd.read_csv(csv_file)
    top_communities = community_data.nlargest(5, 'Number of Nodes').reset_index(drop=True)

    for rank, row in top_communities.iterrows():
        target_community_id = row['Community ID']
        all_nodes = set(map(int, row['Nodes'].split(',')))
        filtered_edges = filter_edge_list(edge_list_file, all_nodes)

        visualize_5_rows_with_AI_colors(
            filtered_edges,
            community_data,
            target_community_id,
            rank + 1,  
            save_folder,
            vote_results_pkl,
            out_deg_max=100
        )

if __name__ == "__main__":
    main()

Pred Data

In [ ]:
def main():
    # Bad
    csv_file = str(OUTPUT_DIR / 'pred/metadata/louvain/best_louvain_reso_1.8.csv')
    edge_list_file = str(OUTPUT_DIR / 'pred/metadata/edge_list.txt')
    save_folder = str(OUTPUT_DIR / 'pred/metadata/louvain/ai_network_diagram')
    vote_results_pkl = str(DATA_DIR / 'pred/dataset/pkl/vote_results.pkl')

    community_data = pd.read_csv(csv_file)
    top_communities = community_data.nlargest(5, 'Number of Nodes').reset_index(drop=True)

    for rank, row in top_communities.iterrows():
        target_community_id = row['Community ID']
        all_nodes = set(map(int, row['Nodes'].split(',')))
        filtered_edges = filter_edge_list(edge_list_file, all_nodes)

        visualize_5_rows_with_AI_colors(
            filtered_edges,
            community_data,
            target_community_id,
            rank + 1,  
            save_folder,
            vote_results_pkl,
            out_deg_max=100
        )

if __name__ == "__main__":
    main()